# `utils.py` — Added Functions: Explanation & Demo

This notebook explains and demonstrates the functions added to `utils.py`
(all marked with `# ADDED`).  Each section covers one conceptual block.

---
**Author:** student  
**Module:** `utils.py` — Section 1 (from-scratch metrics) + Section 2 (features) + Section 3-4 (helpers)

In [ ]:
import numpy as np
import pandas as pd
from utils import (
    jaccard_similarity,
    char_bigram_dice,
    lcs_ratio,
    cosine_similarity_tfidf_batch,
    get_handcrafted_features,
    get_combined_features,
    save_object,
    load_object,
    evaluate_model,
)

---
## 1. `jaccard_similarity(str1, str2)`

**Motivation:** Duplicate questions tend to share many of the same words.  
Jaccard measures how much two sets overlap:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

Returns 0 (no overlap) to 1 (identical sets).  
**Implemented from scratch** using Python sets.

In [ ]:
pairs = [
    ("What is the best way to learn Python?",  "What is the best way to learn Python?"),  # identical
    ("How do I learn Python?",                 "What is the best way to learn Python?"),  # partial
    ("What is the capital of France?",          "How many goals did Messi score in 2023?"),# no overlap
]
for q1, q2 in pairs:
    print(f"{jaccard_similarity(q1, q2):.3f}  |  {q1!r}  /  {q2!r}")

**Limitation:** purely lexical — synonyms ('automobile' vs 'car') score 0.

---
## 2. `char_bigram_dice(str1, str2)`

**Motivation:** Character-level overlap is robust to typos and morphological
variation.  The Sørensen–Dice coefficient on character bigrams is a classic
spell-checker metric:

$$D(A, B) = \frac{2\,|A \cap B|}{|A| + |B|}$$

where $A$ and $B$ are *multisets* of character bigrams.  
**Implemented from scratch** using `collections.Counter`.

In [ ]:
pairs_dice = [
    ("machine learning", "machine learning"),
    ("machine learning", "machine learnin"),   # one char missing
    ("neural network",   "neuron networks"),    # partial match
    ("cat",              "dog"),
]
for s1, s2 in pairs_dice:
    print(f"{char_bigram_dice(s1, s2):.3f}  |  {s1!r}  vs  {s2!r}")

---
## 3. `lcs_ratio(str1, str2)`

**Motivation:** The Longest Common Subsequence (LCS) is order-aware — it
rewards shared words *in the same relative order*, making it useful for
detecting reordered paraphrases.

$$\text{lcs\_ratio}(A, B) = \frac{\text{LCS}(A, B)}{\max(|A|, |B|)}$$

**Implemented from scratch** with a DP table (word level).  
Complexity: O(m·n) — suitable for small batches / demonstration.

In [ ]:
pairs_lcs = [
    ("the quick brown fox",     "the quick brown fox"),
    ("the quick brown fox",     "quick the brown fox"),  # reordered → lower
    ("how to learn Python",     "best way to learn Python fast"),
    ("what is machine learning","what is deep learning"),
]
for s1, s2 in pairs_lcs:
    print(f"{lcs_ratio(s1, s2):.3f}  |  {s1!r}  /  {s2!r}")

> **Note:** Because of its O(m·n) complexity, `lcs_ratio` is not used in the
> main feature pipeline for the full 300 k-sample dataset.  It is provided
> as a self-contained, reusable function in `utils.py`.

---
## 4. `cosine_similarity_tfidf_batch(df, tfidf_vectorizer)`

**Motivation:** TF-IDF down-weights frequent words (what, is, the), giving
more weight to semantically important terms.  Cosine similarity on TF-IDF
vectors is a strong baseline for document similarity.

$$\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|\,\|\mathbf{v}\|}$$

**Implemented from scratch** directly on the sparse matrices —
fully vectorised (no Python loops):

```python
dot   = (X_q1 * X_q2).sum(axis=1)     # element-wise product, row sum
norm1 = sqrt(X_q1^2 .sum(axis=1))
norm2 = sqrt(X_q2^2 .sum(axis=1))
cos   = dot / (norm1 * norm2)          # broadcast
```

In [ ]:
import sklearn.feature_extraction.text

demo_df = pd.DataFrame({
    "question1": [
        "What is machine learning?",
        "How do I train a neural network?",
        "What is the capital of France?",
    ],
    "question2": [
        "What is machine learning?",
        "How can I train deep learning models?",
        "Who won the World Cup in 2022?",
    ],
})

demo_tfidf = sklearn.feature_extraction.text.TfidfVectorizer()
all_texts = list(demo_df["question1"]) + list(demo_df["question2"])
demo_tfidf.fit(all_texts)

sims = cosine_similarity_tfidf_batch(demo_df, demo_tfidf)
for i, row in demo_df.iterrows():
    print(f"{sims[i, 0]:.3f}  |  {row.question1!r}  /  {row.question2!r}")

---
## 5. `get_handcrafted_features(df, tfidf_vectorizer)`

Bundles features 0–4 into a single `(n_samples, 5)` numpy array:

| col | Feature | Formula |
|-----|---------|--------|
| 0 | Jaccard | $|A\cap B|/|A\cup B|$ |
| 1 | Length ratio | $\min(l_1,l_2)/\max(l_1,l_2)$ |
| 2 | Common-word F1 | $2|A\cap B|/(|A|+|B|)$ |
| 3 | Char-bigram Dice | $2|bg_1\cap bg_2|/(|bg_1|+|bg_2|)$ |
| 4 | TF-IDF cosine | $\cos(\mathbf{u},\mathbf{v})$ |

In [ ]:
feats = get_handcrafted_features(demo_df, demo_tfidf)
cols = ["jaccard", "len_ratio", "common_f1", "dice_bigram", "tfidf_cosine"]
display(pd.DataFrame(feats, columns=cols).round(3))

---
## 6. `get_combined_features(df, count_vectorizer, tfidf_vectorizer)`

Horizontally stacks the sparse BoW matrix from `get_features_from_df`
with the dense handcrafted features, producing a single sparse matrix
consumed by the improved Logistic Regression.

In [ ]:
# Quick shape check (vectorizers fitted on demo data for illustration)
import sklearn.feature_extraction.text as skt
demo_cv = skt.CountVectorizer()
demo_cv.fit(all_texts)

X_combined = get_combined_features(demo_df, demo_cv, demo_tfidf)
print(f"Combined shape: {X_combined.shape}")
print(f"  BoW vocab:         {len(demo_cv.vocabulary_)} features × 2 questions")
print(f"  Handcrafted feats: 5")

---
## 7. `save_object` / `load_object`

Thin wrappers around `pickle` for persisting any Python object (fitted
sklearn estimator, numpy array, etc.) to disk.  Used in `train_models.ipynb`
to save vectorizers and classifiers.

In [ ]:
import os
os.makedirs("_demo_models", exist_ok=True)

save_object(demo_cv, "_demo_models/demo_cv.pkl")
reloaded = load_object("_demo_models/demo_cv.pkl")
print("Reloaded vocabulary size:", len(reloaded.vocabulary_))

# Cleanup
import shutil
shutil.rmtree("_demo_models")

---
## 8. `evaluate_model(clf, X, y, model_name, split_name)`

Returns a dict with **ROC-AUC**, **Precision**, **Recall** and **F1** — all
required by the deliverable specification — for a fitted classifier on a
given feature matrix and label vector.  Used in `reproduce_results.ipynb`.

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np

# Toy binary classification to illustrate the output format
rng = np.random.default_rng(42)
X_toy = rng.standard_normal((200, 4))
y_toy = (X_toy[:, 0] + rng.standard_normal(200) > 0).astype(int)

clf_toy = LogisticRegression(random_state=0).fit(X_toy, y_toy)
result  = evaluate_model(clf_toy, X_toy, y_toy, model_name="toy_LR", split_name="train")
display(pd.DataFrame([result]))